### Импорты

In [ ]:
from kafka import KafkaConsumer
import xml.etree.ElementTree as ET
import csv
import pandas as pd
from datetime import datetime
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import os
from matplotlib.ticker import MaxNLocator

### Функции

In [ ]:
from kafka import KafkaConsumer
import xml.etree.ElementTree as ET
import csv


def load_logs_df_from_kafka(broker,topic,ns_uri,filename):
    #тэги в XML
    TAGS = {
        'event': 'event',
        'username': 'username',
        'is_success': 'is-success',            # mapping: is-success -> is_success
        'assembly_timestamp': 'assembly-timestamp',
        'timestamp': 'timestamp'
    }

    def get_text(elem, tag):
        child = elem.find(f'{{{NS_URI}}}{tag}')
        return child.text if child is not None else ''

    consumer = KafkaConsumer(
        topic,
        bootstrap_servers=[broker],
        auto_offset_reset='earliest',
        enable_auto_commit=False,
        value_deserializer=lambda m: m.decode('utf-8', errors='ignore'),
        consumer_timeout_ms=10000  # остановится если нет сообщений
    )

    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['event','username','is_success','assembly_timestamp','timestamp'])

        for msg in consumer:
            xml = msg.value.strip()
            try:
                root = ET.fromstring(xml)
            except ET.ParseError:
                # можно логировать ошибку и продолжать
                continue

            row = []
            for col, tag in TAGS.items():
                row.append(get_text(root, tag))
            writer.writerow(row)

    consumer.close()

In [ ]:
def get_sorted_log_df(filename,st_date,end_date,except_users,except_plugins):
    
    regex_pattern = '|'.join(except_users)

    format = f"%Y-%m-%d"
    df = pd.read_csv('plugins_usage.csv')

    df['assembly_timestamp'] = df['assembly_timestamp'].apply(lambda x: datetime.strptime(x[:10],format))
    df['timestamp'] = df['timestamp'].apply(lambda x: datetime.strptime(x[:10],format))
    df = df[df['timestamp'].between(st_date,end_date)]
    df = df[df['username'].str.contains(regex_pattern, case=False, regex=True) == False]

    df = df[df['event'].isna() == False]
    df['Блок плагинов'] = df['event'].str.split('_').apply(lambda x: x[0])
    df['Блок плагинов'] = np.where(df['Блок плагинов'].str.contains("Префаб") == True,"Префаб",df['Блок плагинов'])
    df['Блок плагинов'] = np.where(df['Блок плагинов'].str.contains("МП") == True,"Мастерплан",df['Блок плагинов'])
    df['Неделя'] = df['timestamp'].dt.isocalendar().week
    df = df[df['Блок плагинов'].isin(except_plugins) == False]

    return df
    

In [ ]:
except_users = [
            "a.akulinushkina"
            ,"a.filimonov"
            ,"d.usov"
            ,"e.zapretilin"
            ,"i.neustroeva"
            ,"m.bylinkin"
            ,"p.desyatova"
            ,"y.kucher"
            ,"yu.kremenetskaya"
            ,"d.danovskaya"
            ,"u.kartasheva"
            ,"v.semenovykh"
            ,"d.shljapenkova"
            ,"BIM"
            ,"n.khabarov"
            ,"an.goncharuk"
            ,"m.andreichenko"
            ,"a.barbakadze"
            ,"m.nosikov"
            ,"d.ishmatova"
            ,"o.erokhova"
            ,'l.gulkova'
            ,'a.mikhailovskii'
            ,'o.khramtsova'
            ,"a.kateranova"
            ,"a.sivkova"
            ]
except_plugins = ["ConsoleTest","Параметры",'Установочник']

In [ ]:
def group_by_cols(df,cols):
    plug_usage = df.groupby(cols,as_index=False).agg(Кол_во_запусков=('event','count')
                                                            ,Кол_во_человек=('username','nunique')).sort_values('Кол_во_запусков',ascending=False)
    plug_usage = plug_usage.rename({"Кол_во_запусков":"Количество запусков"
    ,"Кол_во_человек":"Количество человек"}
    ,axis=1)
    return plug_usage

In [ ]:
def save_barplot_by_gr_df(gr_df,x_col,y_col,title,subdir):
    # Создаем фигуру с белым фоном
    fig, ax = plt.subplots(figsize=(12, 8), facecolor='white',dpi=600)
    ax.set_facecolor('white')

    bar_color = '#C1DC8B'

    # Создаем barplot с зеленым цветом как на скриншоте
    sns.barplot(
        data=gr_df.sort_values(x_col,ascending=False),
        y=y_col,
        x=x_col,
        orient='h',
        ax=ax,
        color=bar_color  # Зеленый цвет как на скриншоте (seagreen)
    )

    # Добавляем подписи на бары (серые)
    for container in ax.containers:
        ax.bar_label(
            container,
            fmt='%.0f',
            padding=5,
            color='#666666',  # Серый цвет подписей
            fontsize=10,
            fontweight='normal'
        )

    # Добавляем вертикальные серые линии (сетку)
    ax.xaxis.grid(
        True,
        which='major',
        color='#E0E0E0',  # Светло-серый цвет линий
        linestyle='-',
        linewidth=0.5,
        alpha=0.7
    )
    ax.set_axisbelow(True)  # Сетка под барами

    # Настраиваем оси
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#CCCCCC')
    ax.spines['bottom'].set_color('#CCCCCC')

    # Добавляем название графика
    ax.set_title(
        title,
        fontsize=16,
        fontweight='bold',
        pad=20,
        color='#333333'
    )

    # Настраиваем подписи осей
    ax.set_xlabel(x_col, fontsize=12, color='#555555', labelpad=10)
    ax.set_ylabel(y_col, fontsize=12, color='#555555', labelpad=10)

    # Настраиваем тики
    ax.tick_params(axis='both', colors='#666666', labelsize=10)

    # Автоматическая подгонка layout
    plt.tight_layout()

    #Сохранение
    base_dir = os.path.dirname(os.getcwd()) 
    plots_dir = os.path.join(base_dir, 'Plots',subdir)
    
    if not os.path.exists(plots_dir):
        os.makedirs(plots_dir)
    
    title = title.replace(' ', '_').replace('/', '_').replace(r'"',"")
    save_path = os.path.join(plots_dir, f"{title}.png")
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    print(f"График сохранен по пути: {save_path}")
    plt.close()

In [ ]:
def save_plot(filename: str):
    plots_dir = "Plots"
    if not os.path.exists(plots_dir):
        os.makedirs(plots_dir)
    plt.savefig(os.path.join(plots_dir, filename), dpi=300, bbox_inches='tight')
    print(f"График сохранен: {os.path.join(plots_dir, filename)}")
    plt.close() # Закрываем фигуру, чтобы освободить память

def plot_each_plugin_separately(df: pd.DataFrame,col_name):
    """
    Создает и сохраняет отдельный файл графика для каждого плагина.
    """
    # df = prepare_data(df)
    plugins = df[col_name].unique()
    bar_color = '#C1DC8B'  # Ваш цвет
    
    for plugin in plugins:
        # Фильтруем данные для текущего плагина
        subset = df[df[col_name] == plugin]
        
        plt.figure(figsize=(8, 4))
        
        # Добавляем color=bar_color для линии и маркеров
        ax = sns.lineplot(
            data=subset, 
            x='Дата', 
            y='Количество запусков', 
            marker='o', 
            color=bar_color
        )
        
        # Настройка оси X: только целые числа
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        
        # Поворот подписей оси X на 45 градусов
        plt.xticks(rotation=45, ha='right')
        
        plt.title(f'Динамика: {plugin}')
        plt.grid(True)
        plt.xlabel('Дата')
        plt.ylabel('Количество запусков')
        
        # Создаем безопасное имя файла
        name = plugin.replace(' ', '_').replace('/', '_').replace(r'"',"")
        safe_filename = f"dynamics_{name}.png"
        save_plot(safe_filename)

### Загрузка данных

In [ ]:
broker = 'cld-kfk-04.brusnika.ltd'
topic = 'bru.revit-plugin.logs'
ns_uri = 'http://schema.brusnika.tech/mdm/oa/pc/design/revit-plugin/revit-plugin-logs'
filename = 'plugins_usage.csv'

st_date = datetime(2025,9,29)
end_date = datetime(2026,4,2)

load_logs_df_from_kafka(broker,topic,ns_uri,filename) #Грузим csv из кафки
df = get_sorted_log_df(filename,st_date,end_date,except_users,except_plugins)   #Загружаем датафрейм
df.head()


### Статистика использования

In [ ]:
gr_by_plugins = group_by_cols(df,['Блок плагинов'])
save_barplot_by_gr_df(gr_by_plugins,'Количество запусков','Блок плагинов','Распределение запусков по блокам плагинов',"Общие")
save_barplot_by_gr_df(gr_by_plugins,'Количество человек','Блок плагинов','Распределение человек по блокам плагинов',"Общие")

In [ ]:
#Сохранение графиков по кнопкам внутри блоков плагинов
for plug_name in df['Блок плагинов'].unique():
    gr_by_plugins = group_by_cols(df[df['Блок плагинов'] == plug_name],['event'])
    save_barplot_by_gr_df(gr_by_plugins,'Количество запусков','event',f'Распределение запусков по блоку {plug_name}','Распределение по кнопкам')

In [ ]:
#Сохранение графиков по пользователям внутри блоков плагинов
for plug_name in df['Блок плагинов'].unique():
    gr_by_plugins = group_by_cols(df[df['Блок плагинов'] == plug_name],['username'])
    save_barplot_by_gr_df(gr_by_plugins,'Количество запусков','username',f'Распределение запусков по блоку {plug_name}','Распределение по пользователям')

In [ ]:
#Сохранение графиков по пользователям и кнопкам внутри блоков плагинов
for plug_name in df['Блок плагинов'].unique():
    gr_by_plugins = group_by_cols(df[df['Блок плагинов'] == plug_name],['username','event'])
    gr_by_plugins.loc[:,'Пользователь_Кнопка'] = gr_by_plugins['username'] + "_" + gr_by_plugins['event']
    save_barplot_by_gr_df(gr_by_plugins,'Количество запусков','Пользователь_Кнопка',f'Распределение запусков по блоку {plug_name}','Распределение по пользователям и кнопкам')

In [ ]:
# need_events = [
#     "Справочники_Справочник помещений",
#     "Справочник АР_Проверка имен уровней",
#     "Справочник АР_Справочник имен уровней",
#     "Инструменты АР_Черновая отделка пола",
#     "Инструменты АР_Черновая отделка стен",
#     "Инструменты АР_Колонны в стены",
#     "Инструменты АР_Отменить соединение стен",
#     "Инструменты АР_Ограждение в границы помещения",
#     "Удаление линий на фасаде",
#     "Инструменты АР_Выбрать помещения",
#     "Инструменты АР_Выбрать окна и витражи",
#     "Инструменты АР_Выбрать окна/стены/перекрытия",
#     "Инструменты АР_Проверка типовых этажей",
#     "Параметры АР_Заполнить параметр Оси",
#     "Параметры АР_Заполнить BRU_ВитражТип",
#     "Параметры АР_Рассчитать Площадь Армирования для стен",
#     "Параметры АР_Заполнить Группу модели для стен АР",
#     "Инструменты Концепция_Заполнить Ось по умолчанию",
#     "Инструменты Концепция_Назначить параметры для откосов (стены и перекрытия)",
#     "Инструменты АР_Создание розеток рядом с мебелью",
#     "Инструменты ВИС_Создание выключателей рядом с дверьми",
#     "Инструменты ВИС_Создание коробок освещения в жилых помещениях"
# ]

# #Сохранение графиков по пользователям и кнопкам внутри блоков плагинов ДЛЯ ЮЛИ
# for plug_name in need_events:
#     gr_by_plugins = group_by_cols(df[df['event'] == plug_name],['username','event'])
#     if not len(gr_by_plugins):
#         continue
#     gr_by_plugins.loc[:,'Пользователь_Кнопка'] = gr_by_plugins['username'] + "_" + gr_by_plugins['event']
#     save_barplot_by_gr_df(gr_by_plugins,'Количество запусков','Пользователь_Кнопка',f'Распределение запусков по блоку {plug_name}','Распределение по пользователям и кнопкам ДЛЯ ЮЛИ')

In [ ]:
gr_by_plugins = group_by_cols(df,['event','timestamp'])
# plot_each_plugin_separately(plug_usage,'Блок плагинов') #Для блоков плагинов
#plot_each_plugin_separately(button_usage,'event') #Для кнопок

#Для оценки пульса плагина

### Какие кнопки не используются

In [ ]:
last_plugins = df.groupby('event').agg(
    Кол_во_запусков=('username','count'),
    Крайний_запуск=('timestamp','max')
    )
last_plugins = last_plugins.sort_values('Кол_во_запусков',ascending=True)
last_plugins.head(2)